# Semana 9: RQ4, inventário das referências candidatas

Seção 2 de `prereg/plano_rq4.md`. **Este notebook não roda nenhum modelo e não lê nenhuma saída de modelo.** Ele baixa as referências candidatas de 06/05/2024 e faz um inventário de cada camada: data no nome, número de feições, classes, e área dentro de cada tile (22JDM e 22JCN). Nada é rasterizado nem comparado com nada ainda. O inventário serve para decidir, pela regra do plano, quais camadas entram.

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, clona o repositório |
| 2 | Pede ao Copernicus EMS os metadados da ativação EMSR720 (áreas, produtos, imagens de origem e datas) e grava a resposta |
| 3 | Baixa o pacote da ativação EMSN194 (dados de 42 MB, relatório e KML das áreas) |
| 4 | Baixa a base da UFRGS no Zenodo (registro 14617475, versão 1.4): só o GeoPackage principal (218 MB) e o glossário, conferidos pelo MD5 do Zenodo |
| 5 | Inventário de todas as camadas e registro da rodada |

Abra o Colab logada na conta do mestrado (não precisa de GPU). Leva uns 10 minutos. Se alguma célula der erro, pare e me mande a saída inteira. A célula 2 pode não achar os metadados do EMSR720 (o portal muda de endereço às vezes); nesse caso ela avisa e segue, sem erro.

Os arquivos baixados ficam em `dados/rs2024/referencias` e não são baixados de novo numa segunda rodada. A saída de cada rodada vai para uma pasta nova (`rq4_referencias`, depois `rq4_referencias_2`, e assim por diante); a célula 1 mostra qual.

Depois da célula 5, me mande **todos os arquivos** da pasta de saída desta rodada (são pequenos).

In [ ]:
# Célula 1 de 5: Drive e repositório
import os, re, json, time, hashlib, glob, shutil, subprocess, sys, zipfile
import requests

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

REF = f'{RAIZ}/dados/rs2024/referencias'
SAIDA_BASE = f'{RAIZ}/resultados/semana9/rq4_referencias'
LOCAL = '/content/referencias'
# Nunca sobrescreve: se a pasta já tem arquivos de uma rodada anterior, usa a próxima livre (_2, _3, ...).
SAIDA, n = SAIDA_BASE, 1
while os.path.isdir(SAIDA) and os.listdir(SAIDA):
    n += 1
    SAIDA = f'{SAIDA_BASE}_{n}'
print('Saída desta rodada:', SAIDA.replace(MEUDRIVE, 'MyDrive'))
for d in (REF, SAIDA, LOCAL):
    os.makedirs(d, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyogrio'], check=True)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
import inventario_referencias as I
ENTRADAS = f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas.json'
CAIXAS = I.caixas_tiles(ENTRADAS)

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def md5_arquivo(caminho):
    h = hashlib.md5()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def baixar(url, destino):
    # Baixa para o disco local e só depois copia para o Drive; não baixa de novo se o arquivo já está no Drive.
    no_drive = f'{REF}/{os.path.relpath(destino, LOCAL)}'
    if os.path.exists(no_drive):
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        if not os.path.exists(destino):
            shutil.copy(no_drive, destino)
        return destino
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(destino + '.parcial', 'wb') as f:
            for bloco in r.iter_content(1 << 20):
                f.write(bloco)
    os.replace(destino + '.parcial', destino)
    os.makedirs(os.path.dirname(no_drive), exist_ok=True)
    shutil.copy(destino, no_drive)
    return destino

ARQUIVOS = {}     # nome -> {url, bytes, sha256}
print(f'Repositório no commit {commit[:7]}. Tiles:', CAIXAS)

In [ ]:
# Célula 2 de 5: metadados do EMSR720
URLS_EMSR720 = [
    'https://rapidmapping.emergency.copernicus.eu/backend/dashboard-api/public-activations-info/?code=EMSR720',
    'https://rapidmapping.emergency.copernicus.eu/backend/dashboard-api/public-activations/?code=EMSR720',
]
EMSR720_OK = False
for i, url in enumerate(URLS_EMSR720, 1):
    try:
        r = requests.get(url, timeout=60, headers={'Accept': 'application/json'})
        print(f'{url}\n   resposta {r.status_code}, {len(r.content)} bytes')
        if r.status_code == 200 and r.content.strip()[:1] in (b'{', b'['):
            with open(f'{SAIDA}/emsr720_api_{i}.json', 'wb') as f:
                f.write(r.content)
            EMSR720_OK = True
    except Exception as erro:
        print(f'{url}\n   falhou: {type(erro).__name__}')
if EMSR720_OK:
    print('Metadados do EMSR720 gravados. Eu leio e digo quais produtos baixar.')
else:
    print('Não achei os metadados do EMSR720 por aqui. Sem problema: siga para a célula 3.')

In [ ]:
# Célula 3 de 5: pacote do EMSN194
EMSN194 = {
    'EMSN194_Reporting.zip': 'https://riskandrecovery.emergency.copernicus.eu/download/EMSN194/EMSN194_Reporting.zip/',
    'EMSN194_GeospatialData.zip': 'https://riskandrecovery.emergency.copernicus.eu/download/EMSN194/EMSN194_GeospatialData.zip/',
    'EMSN194_AOIs.kml': 'https://riskandrecovery.emergency.copernicus.eu/kml/EMSN194/',
}
for nome, url in EMSN194.items():
    caminho = baixar(url, f'{LOCAL}/emsn194/{nome}')
    ARQUIVOS[f'emsn194/{nome}'] = {'url': url, 'bytes': os.path.getsize(caminho), 'sha256': sha256_arquivo(caminho)}
    print(f'{nome}: {os.path.getsize(caminho) / 1e6:.2f} MB')
for nome in ('EMSN194_Reporting.zip', 'EMSN194_AOIs.kml'):
    shutil.copy(f'{LOCAL}/emsn194/{nome}', f'{SAIDA}/{nome}')
GEO_EMSN194 = f'{LOCAL}/emsn194/geodata'
with zipfile.ZipFile(f'{LOCAL}/emsn194/EMSN194_GeospatialData.zip') as z:
    nomes = z.namelist()
    z.extractall(GEO_EMSN194)
# pacotes internos (zip dentro do zip) também são abertos
for interno in glob.glob(f'{GEO_EMSN194}/**/*.zip', recursive=True):
    with zipfile.ZipFile(interno) as z:
        z.extractall(interno[:-4])
with open(f'{SAIDA}/emsn194_conteudo.txt', 'w') as f:
    f.write('\n'.join(sorted(os.path.relpath(p, GEO_EMSN194) for p in glob.glob(f'{GEO_EMSN194}/**/*', recursive=True))))
print(f'EMSN194: {len(nomes)} itens no pacote de dados; lista em emsn194_conteudo.txt')

In [ ]:
# Célula 4 de 5: base da UFRGS no Zenodo
ZENODO = 'https://zenodo.org/api/records/14617475'
r = requests.get(ZENODO, timeout=60)
r.raise_for_status()
with open(f'{SAIDA}/zenodo_14617475.json', 'wb') as f:
    f.write(r.content)
rec = r.json()
print('Zenodo:', rec['metadata'].get('title'), '| versão', rec['metadata'].get('version'), '| publicado', rec['metadata'].get('publication_date'))
arquivos = {a['key']: a for a in rec['files']}
principal = [k for k in arquivos if re.fullmatch(r'cheias_rhguaiba_2024_db_v\d+\.gpkg', k)]
glossario = [k for k in arquivos if re.fullmatch(r'cheias_rhguaiba_2024_db_v\d+_glossario\.csv', k)]
assert len(principal) == 1 and len(glossario) == 1, (principal, glossario)
for chave in principal + glossario:
    a = arquivos[chave]
    url = a.get('links', {}).get('self') or f'https://zenodo.org/records/14617475/files/{chave}?download=1'
    caminho = baixar(url, f'{LOCAL}/ufrgs/{chave}')
    md5 = a['checksum'].split(':', 1)[1]
    assert md5_arquivo(caminho) == md5, f'{chave}: MD5 diferente do informado pelo Zenodo. Pare e me mande esta saída.'
    ARQUIVOS[f'ufrgs/{chave}'] = {'url': url, 'bytes': os.path.getsize(caminho), 'sha256': sha256_arquivo(caminho), 'md5_zenodo': md5}
    print(f'{chave}: {os.path.getsize(caminho) / 1e6:.1f} MB, MD5 confere')
GPKG_UFRGS = f'{LOCAL}/ufrgs/{principal[0]}'
shutil.copy(f'{LOCAL}/ufrgs/{glossario[0]}', f'{SAIDA}/{glossario[0]}')

In [ ]:
# Célula 5 de 5: inventário e registro da rodada
import warnings
warnings.filterwarnings('ignore')
t0 = time.time()
linhas, valores = I.inventariar([('ufrgs', GPKG_UFRGS), ('emsn194', GEO_EMSN194)], CAIXAS, base=LOCAL)
I.escrever(linhas, valores, CAIXAS, SAIDA)
print(f'{len(linhas)} camadas inventariadas em {time.time() - t0:.0f} s.\n')
chave = re.compile(r'inunda|agua|água|lama|flood|water|obs|extent|ada|polder|hec', re.I)
for l in linhas:
    if chave.search(f"{l.get('arquivo', '')} {l.get('camada', '')}") or l.get('erro'):
        nome = f"{os.path.basename(l.get('arquivo', ''))} :: {l.get('camada', '')}"
        datas = ';'.join(sorted(set(filter(None, (l.get('datas_no_nome', '') + ';' + l.get('datas_no_arquivo', '')).split(';')))))
        print(f"{l['fonte']:8s} {nome[-90:]:90s} datas {datas:22s} "
              f"n {l.get('n_feicoes', '')!s:>6s}  22JDM {l.get('area_km2_22JDM', '')!s:>9s} km²  22JCN {l.get('area_km2_22JCN', '')!s:>9s} km²  {l.get('erro', '')}")

from importlib.metadata import version
INFO = {'plano': 'prereg/plano_rq4.md, secao 2', 'commit_repositorio': commit,
        'inventario_sha256': sha256_arquivo(f'{REPO_GIT}/analysis/rq4/inventario_referencias.py'),
        'arquivos': ARQUIVOS, 'emsr720_metadados': EMSR720_OK,
        'versoes': {p: version(p) for p in ('geopandas', 'pyogrio', 'shapely')},
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA}/rodada.json', 'w'), indent=2, ensure_ascii=False)
print('\nArquivos para me mandar:', sorted(os.listdir(SAIDA)))
print('Nenhum modelo foi rodado.')